In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

analysis_data = pd.read_csv("f1_analysis_data.csv")
analysis_data

,Unnamed: 0.1,Unnamed: 0,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,...,IsAccurate,Year,Round,Session,fuel_kg,fuel_adjustment,fuel_effect,lap_time_s,fuel_delta_to_55,lap_time_fuel_std_55kg
0,1,1,0 days 01:05:53.220000,VER,1,0 days 00:01:37.880000,2.0,1.0,NaN,NaN,...,True,2022,1,Race,108.049107,3.241473,0.03,97.880,1.591473,96.288527
1,2,2,0 days 01:07:31.577000,VER,1,0 days 00:01:38.357000,3.0,1.0,NaN,NaN,...,True,2022,1,Race,106.098214,3.182946,0.03,98.357,1.532946,96.824054
2,3,3,0 days 01:09:10.143000,VER,1,0 days 00:01:38.566000,4.0,1.0,NaN,NaN,...,True,2022,1,Race,104.147321,3.124420,0.03,98.566,1.474420,97.091580
3,4,4,0 days 01:10:49.020000,VER,1,0 days 00:01:38.877000,5.0,1.0,NaN,NaN,...,True,2022,1,Race,102.196429,3.065893,0.03,98.877,1.415893,97.461107
4,5,5,0 days 01:12:27.960000,VER,1,0 days 00:01:38.940000,6.0,1.0,NaN,NaN,...,True,2022,1,Race,100.245536,3.007366,0.03,98.940,1.357366,97.582634
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91975,103416,103416,0 days 02:13:00.598000,PIA,81,0 days 00:01:34.588000,46.0,2.0,NaN,NaN,...,True,2025,22,Race,9.668367,0.193367,0.02,94.588,-0.906633,95.494633
91976,103417,103417,0 days 02:14:34.684000,PIA,81,0 days 00:01:34.086000,47.0,2.0,NaN,NaN,...,True,2025,22,Race,7.438776,0.148776,0.02,94.086,-0.951224,95.037224
91977,103418,103418,0 days 02:16:08.849000,PIA,81,0 days 00:01:34.165000,48.0,2.0,NaN,NaN,...,True,2025,22,Race,5.209184,0.104184,0.02,94.165,-0.995816,95.160816
91978,103419,103419,0 days 02:17:43.315000,PIA,81,0 days 00:01:34.466000,49.0,2.0,NaN,NaN,...,True,2025,22,Race,2.979592,0.059592,0.02,94.466,-1.040408,95.506408


In [2]:
analysis_data = analysis_data.sort_values(
    ["Year", "Round", "Driver", "Stint", "LapNumber"]
)

analysis_data["lap_in_stint"] = (
    analysis_data
    .groupby(["Year", "Round", "Driver", "Stint"])
    .cumcount() + 1
)

analysis_data
print(analysis_data.columns)

Index(['Unnamed: 0.1', 'Unnamed: 0', 'Time', 'Driver', 'DriverNumber',
       'LapTime', 'LapNumber', 'Stint', 'PitOutTime', 'PitInTime',
       'Sector1Time', 'Sector2Time', 'Sector3Time', 'Sector1SessionTime',
       'Sector2SessionTime', 'Sector3SessionTime', 'SpeedI1', 'SpeedI2',
       'SpeedFL', 'SpeedST', 'IsPersonalBest', 'Compound', 'TyreLife',
       'FreshTyre', 'Team', 'LapStartTime', 'LapStartDate', 'TrackStatus',
       'Position', 'Deleted', 'DeletedReason', 'FastF1Generated', 'IsAccurate',
       'Year', 'Round', 'Session', 'fuel_kg', 'fuel_adjustment', 'fuel_effect',
       'lap_time_s', 'fuel_delta_to_55', 'lap_time_fuel_std_55kg',
       'lap_in_stint'],
      dtype='object')


In [3]:
def compute_stint_tyre_deg(df,
                           time_col="lap_time_fuel_std_55kg",
                           lap_in_stint_col="lap_in_stint",
                           min_laps=3):

    results = []

    group_cols = ["Year", "Round", "Driver", "Stint"]

    for keys, g in df.groupby(group_cols):
        g = g.dropna(subset=[lap_in_stint_col, time_col])
        if len(g) < min_laps:
            continue

        x = g[lap_in_stint_col].astype(float).values
        y = g[time_col].astype(float).values

        slope, intercept = np.polyfit(x, y, 1)

        results.append({
            "Year":   keys[0],
            "Round":  keys[1],
            "Driver": keys[2],
            "Stint":  keys[3],
            "slope_s_per_lap": slope,
            "intercept_s":     intercept,
            "n_laps":          len(g),
        })

    return pd.DataFrame(results)

In [4]:
stint_deg = compute_stint_tyre_deg(analysis_data)
stint_deg

,Year,Round,Driver,Stint,slope_s_per_lap,intercept_s,n_laps
0,2022,1,ALB,1.0,0.292018,98.926236,11
1,2022,1,ALB,2.0,0.130616,99.241017,20
2,2022,1,ALB,3.0,0.094098,99.445009,7
3,2022,1,ALB,4.0,0.451848,98.217098,7
4,2022,1,ALO,1.0,0.298793,98.079000,9
...,...,...,...,...,...,...,...
4480,2025,22,SAI,2.0,-0.001470,95.606106,27
4481,2025,22,TSU,2.0,0.045961,96.331213,21
4482,2025,22,TSU,3.0,0.028005,95.826278,22
4483,2025,22,VER,1.0,-0.020392,95.937498,19


In [5]:
compound_per_stint = (
    analysis_data
    .groupby(["Year", "Round", "Driver", "Stint"])["Compound"]
    .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)
    .reset_index()
)

stint_deg = stint_deg.merge(
    compound_per_stint,
    on=["Year", "Round", "Driver", "Stint"],
    how="left"
)

stint_deg

,Year,Round,Driver,Stint,slope_s_per_lap,intercept_s,n_laps,Compound
0,2022,1,ALB,1.0,0.292018,98.926236,11,C3
1,2022,1,ALB,2.0,0.130616,99.241017,20,C2
2,2022,1,ALB,3.0,0.094098,99.445009,7,C2
3,2022,1,ALB,4.0,0.451848,98.217098,7,C3
4,2022,1,ALO,1.0,0.298793,98.079000,9,C3
...,...,...,...,...,...,...,...,...
4480,2025,22,SAI,2.0,-0.001470,95.606106,27,C3
4481,2025,22,TSU,2.0,0.045961,96.331213,21,C3
4482,2025,22,TSU,3.0,0.028005,95.826278,22,C4
4483,2025,22,VER,1.0,-0.020392,95.937498,19,C4


In [13]:
num_drivers = stint_deg["Driver"].nunique() + 1
num_compounds = stint_deg["Compound"].nunique() + 1
# num_years = df["Year"].nunique()
# num_rounds = stint_deg["Round"].nunique()
num_stints = stint_deg["Stint"].nunique() + 1

In [14]:
import tensorflow as tf
from tensorflow.keras import layers

# Numeric features
numeric_inputs = tf.keras.Input(shape=(2,), name="numeric")  # intercept_s, n_laps

# Categorical features (integer encoded beforehand)
driver_input = tf.keras.Input(shape=(1,), name="driver")
compound_input = tf.keras.Input(shape=(1,), name="compound")
year_input = tf.keras.Input(shape=(1,), name="year")
# round_input = tf.keras.Input(shape=(1,), name="round")
stint_input = tf.keras.Input(shape=(1,), name="stint")

# Embeddings
driver_emb = layers.Embedding(num_drivers, 8)(driver_input)
compound_emb = layers.Embedding(num_compounds, 4)(compound_input)
# year_emb = layers.Embedding(num_years, 2)(year_input)
# round_emb = layers.Embedding(num_rounds, 2)(round_input)
stint_emb = layers.Embedding(num_stints, 2)(stint_input)

cat_features = layers.Concatenate()([
    layers.Flatten()(driver_emb),
    layers.Flatten()(compound_emb),
    # layers.Flatten()(year_emb),
    # layers.Flatten()(round_emb),
    layers.Flatten()(stint_emb)
])

# Combine with numeric
x = layers.Concatenate()([numeric_inputs, cat_features])

# MLP
x = layers.Dense(128, activation="relu")(x)
x = layers.Dense(64, activation="relu")(x)
x = layers.Dense(32, activation="relu")(x)
output = layers.Dense(1)(x)  

model = tf.keras.Model(
    # inputs=[numeric_inputs, driver_input, compound_input, year_input, round_input, stint_input],
    inputs=[numeric_inputs, driver_input, compound_input, stint_input],
    outputs=output
)

model.compile(optimizer='adam', loss='mse')


In [18]:
test_data =  stint_deg[stint_deg["Year"] == 2025]
train_data = stint_deg[stint_deg["Year"] != 2025]

train_data.loc[:, "Driver"] = train_data["Driver"].astype(str)
test_data.loc[:, "Driver"]  = test_data["Driver"].astype(str)

train_data.loc[:, "Compound"] = train_data["Compound"].astype(str)
test_data.loc[:, "Compound"]  = test_data["Compound"].astype(str)

train_data.loc[:, "Stint"] = train_data["Stint"].astype(str)
test_data.loc[:, "Stint"]  = test_data["Stint"].astype(str)


# print(test_data)
# print(train_data)

C:\Users\onehi\AppData\Local\Temp\ipykernel_23336\1732987793.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['1.0' '2.0' '3.0' ... '1.0' '2.0' '3.0']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  train_data.loc[:, "Stint"] = train_data["Stint"].astype(str)
C:\Users\onehi\AppData\Local\Temp\ipykernel_23336\1732987793.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['4.0' '6.0' '4.0' ... '3.0' '1.0' '2.0']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  test_data.loc[:, "Stint"]  = test_data["Stint"].astype(str)


In [19]:
def build_mapping(values):
    uniq = sorted(values.unique())
    mapping = {v: i for i, v in enumerate(uniq)}
    unknown_id = len(mapping)  # extra category for unseen
    return mapping, unknown_id

driver_map, driver_unknown = build_mapping(train_data["Driver"])
compound_map, compound_unknown = build_mapping(train_data["Compound"])
stint_map, stint_unknown = build_mapping(train_data["Stint"])


In [20]:
def encode_with_unknown(df, mapping, unknown_id):
    return df.apply(lambda x: mapping.get(x, unknown_id)).astype(int).values.reshape(-1, 1)


In [21]:
def encode(df):
    return {
        "numeric": df[["intercept_s", "n_laps"]].values.astype(float),
        
        "driver": encode_with_unknown(df["Driver"], driver_map, driver_unknown),
        "compound": encode_with_unknown(df["Compound"], compound_map, compound_unknown),
        "stint": encode_with_unknown(df["Stint"], stint_map, stint_unknown),
    }


X_train = encode(train_data)
X_test  = encode(test_data)

y_train = train_data["slope_s_per_lap"].values
y_test  = test_data["slope_s_per_lap"].values

In [27]:
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2958 - val_loss: 0.1601
Epoch 2/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2189 - val_loss: 0.2160
Epoch 3/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3101 - val_loss: 0.1383
Epoch 4/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2429 - val_loss: 0.2437
Epoch 5/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3109 - val_loss: 0.1215
Epoch 6/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2526 - val_loss: 0.1104
Epoch 7/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2066 - val_loss: 0.2095
Epoch 8/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2216 - val_loss: 0.1161
Epoch 9/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2183 - val_loss: 0.1233
Epoch 10/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2052 - val_loss: 0.1838
Epoch 11/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2344 - val_loss: 0.1167


In [28]:
preds = model.predict(X_test)
print(preds[:10])

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 874us/step
[[ 0.01026152]
 [-0.17127742]
 [ 0.00899778]
 [-0.00513805]
 [-0.22193767]
 [-0.00142075]
 [-0.26583445]
 [-0.0052947 ]
 [ 0.01933359]
 [-0.00064327]]


In [29]:
preds = preds.flatten()

In [30]:
import numpy as np

rmse = np.sqrt(np.mean((preds - y_test)**2))
print("Test RMSE:", rmse)

Test RMSE: 0.5070693386296348


In [32]:
print(stint_deg["slope_s_per_lap"].describe())

count    4485.000000
mean       -0.002184
std         0.581657
min       -25.313981
25%        -0.007607
50%         0.050429
75%         0.095853
max         6.014843
Name: slope_s_per_lap, dtype: float64
